In [0]:
catalog_name = 'ecommerce'
df_silver_order_items = spark.table(f'{catalog_name}.bronze.brz_order_items')
df_silver_order_items.show()

In [0]:
df_silver_order_items.printSchema()

In [0]:
from pyspark.sql import functions as F

# Transformation : Drop any duplicates
df_silver_order_items = df_silver_order_items.dropDuplicates(["order_id","item_seq"])

# Transformation : Convert 'Two' -> 2 and cast to integer
df_silver_order_items = df_silver_order_items.withColumn(
    "quantity", 
    F.when(F.col("quantity") == "Two", 2).otherwise(F.col("quantity")).cast("int")
)

# Transformation : Remove and '$' or other symbols from unit_price, keep only numeric
df_silver_order_items = df_silver_order_items.withColumn(
    "unit_price", 
    F.regexp_replace("unit_price", "[$]", "").cast("double")
)

# Transformation : Remove and '$' from discount_pct and cast to double
df_silver_order_items = df_silver_order_items.withColumn(
    "discount_pct", 
    F.regexp_replace("discount_pct", "%", "").cast("double")
)

# Transformation : coupon code processing (convert to lower)
df_silver_order_items = df_silver_order_items.withColumn(
    "coupon_code", 
    F.lower(F.trim(F.col("coupon_code")))
) 

# Transformation : channel processing
df_silver_order_items = df_silver_order_items.withColumn(
    "channel", 
    F.when(F.col("channel") == "web", "Website")
    .when(F.col("channel") == "app", "Mobile")
    .otherwise(F.col("channel")),
)

In [0]:
display(df_silver_order_items)


In [0]:
# Transformation : datatype conversions
# 1) Convert dt (string -> date)
df_silver_order_items = df_silver_order_items.withColumn(
    "dt", 
    F.to_date(F.col("dt"), "yyyy-MM-dd")
)

# 2) Convert order_ts (string -> timestamp)
df_silver_order_items = df_silver_order_items.withColumn(
    "order_ts",
    F.coalesce(F.to_timestamp(F.col("order_ts"), "yyyy-MM-dd HH:mm:ss"), F.to_timestamp(F.col("order_ts"), "yyyy-MM-dd HH:mm"))
)

# 3) Convert item_seq (string -> integer)
df_silver_order_items = df_silver_order_items.withColumn(
    "item_seq", 
    F.col("item_seq").cast("int")
)

# 4) Convert tax_amount (string -> double, strip non-numeric characters)
df_silver_order_items = df_silver_order_items.withColumn(
    "tax_amount", 
    F.regexp_replace("tax_amount", "[^0-9.\-]", "").cast("double")
)

# Transformation : Add processed timestamp
df_silver_order_items = df_silver_order_items.withColumn(
    "ingest_timestamp", 
    F.current_timestamp()
)


In [0]:
display(df_silver_order_items.limit(10))


In [0]:
df_silver_order_items.printSchema()


In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: silver_order_items)
df_silver_order_items.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_order_items")

In [0]:
%sql
DROP TABLE ecommerce.silver.silver_order_items